# Non-Private Neural Network Training

Dataset: **IPBlock Fraud Detection** (Amazon Fraud Dataset Benchmark)

This notebook corresponds to **Section 4 "Non-private Modeling and Evaluation"**
of the paper, applied to the IPBlock fraud detection task. It produces the
Fraud column of Table 1 (baseline non-private NN evaluation metrics).

> **Notes on differences vs. the BankMarketing §4 baseline:**
> - Boruta feature selection is **not** applied here. Per paper §4.1: *"For
>   the IP Blocklist fraud dataset, Boruta was not applied due to the small
>   number of engineered features."*
> - The train/test split is provided by the FDB upstream files; an
>   additional 20% of the training subset is reserved for validation via
>   `validation_split=0.2` in `model.fit`, per paper §4.3.

> **Environment**: run this notebook in the `.ldp` virtual environment
> (TensorFlow 2.18 + Keras 3; see `requirements/ldp.txt`).
> The `.cdp` environment also works (TensorFlow 2.3) since this notebook
> uses no `tensorflow_privacy` ops.

> **Data prerequisites**: the three IPBlock CSVs must exist in
> `../data/raw/`. To produce them, run once from the `.fdb` environment:
> `python -m src.fdb_export.export_data`.

In [1]:
import json
import os
import random
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, confusion_matrix,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Paths
ROOT = Path.cwd().resolve()                 # FraudDetection/notebooks
DATA_RAW = ROOT.parent / "data" / "raw"
FIG_DIR = ROOT.parent / "figures"
RES_DIR = ROOT.parent / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Experiment configuration
TARGET = "EVENT_LABEL"


### Load fraud dataset

In [2]:
# Load fraud dataset and assemble train/test
train_path = DATA_RAW / "ipblock_train.csv"
test_feat_path = DATA_RAW / "ipblock_test_features.csv"
test_labels_path = DATA_RAW / "ipblock_test_labels.csv"

df_train = pd.read_csv(train_path)
df_test_features = pd.read_csv(test_feat_path)
df_test_labels = pd.read_csv(test_labels_path)

# Merge to align test labels with test features
if "EVENT_ID" in df_test_features.columns and "EVENT_ID" in df_test_labels.columns:
    df_test = df_test_features.merge(
        df_test_labels[["EVENT_ID", TARGET]],
        on="EVENT_ID",
        how="left",
    )
else:
    raise ValueError("EVENT_ID is not present in both test tables.")

print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)


Train shape: (172000, 8)
Test shape : (43000, 7)


### Feature engineering

In [3]:
def engineer_features(df):
    df = df.copy()

    # IP -> first two octets
    df["ip"] = df["ip"].astype(str)
    df["ip_1"] = df["ip"].str.split(".").str[0]
    df["ip_2"] = df["ip"].str.split(".").str[1]

    # Timestamp -> hour, day of week, month, weekend indicator
    dt = pd.to_datetime(df["EVENT_TIMESTAMP"])
    df["event_hour"] = dt.dt.hour
    df["event_dow"] = dt.dt.dayofweek
    df["event_month"] = dt.dt.month
    df["event_is_weekend"] = df["event_dow"].isin([5, 6]).astype(int)

    feature_cols = [
        "ip_1",
        "ip_2",
        "ENTITY_TYPE",
        "event_hour",
        "event_dow",
        "event_month",
        "event_is_weekend",
    ]

    X = df[feature_cols].copy()
    y = df[TARGET].astype(int).copy()
    return X, y, feature_cols


X_train_raw, y_train, feature_cols = engineer_features(df_train)
X_test_raw,  y_test,  _           = engineer_features(df_test)

print("X_train_raw:", X_train_raw.shape, "y_train:", y_train.shape)
print("X_test_raw :", X_test_raw.shape,  "y_test :", y_test.shape)


X_train_raw: (172000, 7) y_train: (172000,)
X_test_raw : (43000, 7) y_test : (43000,)


### One-hot encoding

In [4]:
def one_hot_encode_train_test(X_train, X_test, categorical_cols):
    X_train_oh = pd.get_dummies(
        X_train, columns=categorical_cols, drop_first=True, dtype="float32",
    )
    X_test_oh = pd.get_dummies(
        X_test, columns=categorical_cols, drop_first=True, dtype="float32",
    )
    # Align columns so train and test have the same features
    X_test_oh = X_test_oh.reindex(columns=X_train_oh.columns, fill_value=0.0)
    return X_train_oh, X_test_oh


categorical_cols = feature_cols
X_train_oh, X_test_oh = one_hot_encode_train_test(X_train_raw, X_test_raw, categorical_cols)

print("X_train_oh:", X_train_oh.shape)
print("X_test_oh :", X_test_oh.shape)

X_train_full = X_train_oh.values
X_test_full  = X_test_oh.values
y_train_full = y_train.values
y_test_full  = y_test.values


X_train_oh: (172000, 517)
X_test_oh : (43000, 517)


### Initial undersampling (1:1)

In [5]:
def compute_undersample_indices(y, negative_to_positive_ratio=1.0, random_state=42):
    rng = np.random.RandomState(random_state)
    y_arr = np.asarray(y)

    idx_pos = np.where(y_arr == 1)[0]
    idx_neg = np.where(y_arr == 0)[0]

    n_pos = len(idx_pos)
    n_neg = len(idx_neg)

    n_neg_desired = min(n_neg, int(n_pos * negative_to_positive_ratio))

    idx_neg_sample = rng.choice(idx_neg, size=n_neg_desired, replace=False)
    idx_keep = np.concatenate([idx_pos, idx_neg_sample])
    idx_keep.sort()

    return idx_keep


train_idx_keep = compute_undersample_indices(
    y_train_full,
    negative_to_positive_ratio=1.0,
    random_state=SEED,
)

X_train_bal = X_train_full[train_idx_keep]
y_train_bal = y_train_full[train_idx_keep]

print("Train class distribution (original):")
print(pd.Series(y_train_full).value_counts())
print("\nTrain class distribution (balanced):")
print(pd.Series(y_train_bal).value_counts())


Train class distribution (original):
0    159997
1     12003
Name: count, dtype: int64

Train class distribution (balanced):
1    12003
0    12003
Name: count, dtype: int64


### Build model

In [6]:
def build_model(input_dim, units, hidden_layers, dropout_rate, learning_rate):
    model = Sequential()
    model.add(Dense(units, activation="relu", input_shape=(input_dim,)))
    model.add(Dropout(dropout_rate))
    for _ in range(hidden_layers - 1):
        model.add(Dense(units, activation="relu"))
        model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


### Hyperparameter grid search

Grid follows paper §4.2: learning rates `{0.01, 0.001, 0.0001}`,
dropout rates `{0.1, 0.2, 0.3}`, units `{32, 64, 128}`, hidden layers
`{2, 3}`. Each candidate is trained on the balanced training set with
20% reserved for validation (early stopping monitors `val_loss`).

In [7]:
# Hyperparameter grid (paper §4.2)
param_grid = {
    "learning_rate": [0.01, 0.001, 0.0001],
    "dropout_rate":  [0.1, 0.2, 0.3],
    "units":         [32, 64, 128],
    "hidden_layers": [2, 3],
}

input_dim = X_train_bal.shape[1]

best_score = -np.inf
best_params = None
best_model = None

for lr, dr, u, hl in product(
    param_grid["learning_rate"],
    param_grid["dropout_rate"],
    param_grid["units"],
    param_grid["hidden_layers"],
):
    # Build model
    model = build_model(
        input_dim=input_dim,
        units=u,
        hidden_layers=hl,
        dropout_rate=dr,
        learning_rate=lr,
    )

    # Train model with validation split + early stopping
    early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    history = model.fit(
        X_train_bal,
        y_train_bal,
        validation_split=0.2,
        epochs=50,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0,
    )

    # Evaluate on the held-out portion of the validation split (last fold)
    n_val = int(len(X_train_bal) * 0.2)
    X_val = X_train_bal[-n_val:]
    y_val = y_train_bal[-n_val:]
    y_val_prob = model.predict(X_val, verbose=0).ravel()
    val_auc = roc_auc_score(y_val, y_val_prob)

    params = {"learning_rate": lr, "dropout_rate": dr, "units": u, "hidden_layers": hl}
    print(f"params={params}  val_AUC={val_auc:.4f}")

    if val_auc > best_score:
        best_score = val_auc
        best_params = params
        best_model = model

print()
print("Best params :", best_params)
print("Best val AUC:", round(best_score, 4))


c:\Users\danie\OneDrive\Documentos\1 UNIANDES\PAPER DP\differential-privacy-data-analysis\.venv-ldp\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


params={'learning_rate': 0.01, 'dropout_rate': 0.1, 'units': 32, 'hidden_layers': 2}  val_AUC=0.8397


c:\Users\danie\OneDrive\Documentos\1 UNIANDES\PAPER DP\differential-privacy-data-analysis\.venv-ldp\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


params={'learning_rate': 0.01, 'dropout_rate': 0.1, 'units': 32, 'hidden_layers': 3}  val_AUC=0.8380


c:\Users\danie\OneDrive\Documentos\1 UNIANDES\PAPER DP\differential-privacy-data-analysis\.venv-ldp\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

params={'learning_rate': 0.0001, 'dropout_rate': 0.3, 'units': 64, 'hidden_layers': 3}  val_AUC=0.8408


c:\Users\danie\OneDrive\Documentos\1 UNIANDES\PAPER DP\differential-privacy-data-analysis\.venv-ldp\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


params={'learning_rate': 0.0001, 'dropout_rate': 0.3, 'units': 128, 'hidden_layers': 2}  val_AUC=0.8494


c:\Users\danie\OneDrive\Documentos\1 UNIANDES\PAPER DP\differential-privacy-data-analysis\.venv-ldp\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


params={'learning_rate': 0.0001, 'dropout_rate': 0.3, 'units': 128, 'hidden_layers': 3}  val_AUC=0.8343

Best params : {'learning_rate': 0.0001, 'dropout_rate': 0.3, 'units': 64, 'hidden_layers': 2}
Best val AUC: 0.8552


### Final evaluation on the held-out test set

Metrics: accuracy, precision, recall, F1, Type I error, Type II error,
ROC AUC. Threshold for classification is 0.5.

In [8]:
# Predict on test set
y_prob = best_model.predict(X_test_full, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

# Metrics
acc  = accuracy_score(y_test_full, y_pred)
prec = precision_score(y_test_full, y_pred, zero_division=0)
rec  = recall_score(y_test_full, y_pred, zero_division=0)
f1   = f1_score(y_test_full, y_pred, zero_division=0)
auc  = roc_auc_score(y_test_full, y_prob)

# Confusion matrix -> Type I / Type II errors
tn, fp, fn, tp = confusion_matrix(y_test_full, y_pred).ravel()
type_i_pct  = 100.0 * fp / (fp + tn) if (fp + tn) > 0 else 0.0   # FPR
type_ii_pct = 100.0 * fn / (fn + tp) if (fn + tp) > 0 else 0.0   # FNR

print(f"Accuracy        : {acc:.4f}")
print(f"Precision       : {prec:.4f}")
print(f"Recall          : {rec:.4f}")
print(f"F1 Score        : {f1:.4f}")
print(f"Type I  Error % : {type_i_pct:.4f}")
print(f"Type II Error % : {type_ii_pct:.4f}")
print(f"ROC AUC         : {auc:.4f}")


Accuracy        : 0.7344
Precision       : 0.1806
Recall          : 0.7948
F1 Score        : 0.2943
Type I  Error % : 27.0130
Type II Error % : 20.5205
ROC AUC         : 0.8455


### Save results

In [9]:
results_df = pd.DataFrame([{
    "Accuracy":          acc,
    "Precision":         prec,
    "Recall":            rec,
    "F1 Score":          f1,
    "Type I Error (%)":  type_i_pct,
    "Type II Error (%)": type_ii_pct,
    "ROC AUC":           auc,
    "Best Params":       json.dumps(best_params),
}])

out_path = RES_DIR / "neural_network_results.csv"
results_df.to_csv(out_path)
print(f"Results saved to: {out_path}")
results_df


Results saved to: C:\Users\danie\OneDrive\Documentos\1 UNIANDES\PAPER DP\differential-privacy-data-analysis\FraudDetection\results\neural_network_results.csv


,Accuracy,Precision,Recall,F1 Score,Type I Error (%),Type II Error (%),ROC AUC,Best Params
0,0.734395,0.180619,0.794795,0.294347,27.012974,20.520521,0.845508,"{""learning_rate"": 0.0001, ""dropout_rate"": 0.3,..."
